In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from PRODUCTION.calculateEVS import *
from PRODUCTION.pipeline import *
from PRODUCTION.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

### Load Model

In [2]:
# Load split NGBoost models (mean, variance, calibration factor, and isotonic calibrator)
pts_mean_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MEAN_MODEL_PRODUCTION.pkl')
pts_var_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_VAR_MODEL_PRODUCTION.pkl')
calibration_factor = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_FACTOR_PRODUCTION.pkl')

model = (pts_mean_model, pts_var_model, calibration_factor)  
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

print(f"Loaded models with calibration factor: {calibration_factor}")

Loaded models with calibration factor: 4.5


### Load Player Data and Bookmaker Data

In [3]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

dfsData.head()

/var/folders/9q/5_554qsx5z70w9d_vkmvjg0h0000gn/T/ipykernel_52521/1321450873.py:5: DtypeWarning: Columns (33) have mixed types. Specify dtype option on import or set low_memory=False.
  s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
0,Underdog,player_points,Darius Garland,Over,15.5,-137,2025-11-22,2025-11-21T15:26:59Z
1,Underdog,player_points,Darius Garland,Under,15.5,-137,2025-11-22,2025-11-21T15:26:59Z
2,Underdog,player_points,Pascal Siakam,Over,23.5,-137,2025-11-22,2025-11-21T15:26:59Z
3,Underdog,player_points,Pascal Siakam,Under,23.5,-137,2025-11-22,2025-11-21T15:26:59Z
4,Underdog,player_points,Donovan Mitchell,Over,28.5,-137,2025-11-22,2025-11-21T15:26:59Z


### Update projected starting lineups

In [4]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated /Users/alexgonzalez/Documents/NBA-Prop-Predictor/PRODUCTION/teamInfo.py
Updated 18 teams with confirmed lineups


### Top EVs for single bets

In [5]:
singlePTSBookies = usData[(usData['CATEGORY'] == 'player_points') & (usData['BOOKMAKER'] != 'Bovada') & (usData['BOOKMAKER'] != 'BetOnline.ag')]

singleBets = calculateSingleBets(s26, singlePTSBookies, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)



singleBets = singleBets[['NAME', 'BOOKMAKER','LINE', 'PREDICTION', 'SIDE','ODDS','RECOMMENDATION', 'EV$', 'KELLY_FRACTION','SIGMA FLAG']].head(15)
singleBets.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/singleBets.csv', index=False)
singleBets.head(5)

Processing single bets...
Pre-computing predictions for 134 unique players...


,NAME,BOOKMAKER,LINE,PREDICTION,SIDE,ODDS,RECOMMENDATION,EV$,KELLY_FRACTION,SIGMA FLAG
306,Tre Jones,FanDuel,8.5,14.89,Over,-106,1,6.79,0.720,Med
794,Dillon Brooks,BetMGM,16.5,22.80,Over,-115,1,5.52,0.635,High
394,Alperen Sengun,FanDuel,23.5,28.14,Over,-102,1,5.09,0.519,High
4,Bennedict Mathurin,DraftKings,21.5,26.11,Over,-103,1,4.99,0.514,High
742,Kevin Huerter,BetMGM,12.5,16.73,Over,-105,1,4.68,0.491,High


## Top EVs for 2 leg bets

### Underdog picks

In [6]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

underdogPairs = calculate2LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)


underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'PROB 1', 'PROB 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']]
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs

Pre-computing predictions for 112 players...
Processing 103 players...
Generated 4992 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,PROB 1,PROB 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
3261,Tre Jones,Dillon Brooks,8.5,17.5,14.89,22.80,0.864,0.789,over,over,1,10.04,0.502,Med,High
3384,Isaac Okoro,Alperen Sengun,6.5,23.5,11.22,28.14,0.785,0.762,over,over,1,7.59,0.380,Med,High
698,Bennedict Mathurin,Kevin Huerter,21.5,12.5,26.11,16.73,0.761,0.752,over,over,1,6.82,0.341,High,High
31,Darius Garland,Coby White,15.5,20.5,12.47,24.37,0.737,0.751,under,over,0,6.27,0.313,Low,Med
4388,Julius Randle,Lauri Markkanen,22.5,24.5,26.32,28.53,0.715,0.720,over,over,0,5.12,0.256,High,High
4124,Naji Marshall,Keyonte George,10.5,19.5,13.91,23.11,0.713,0.702,over,over,0,4.71,0.236,High,High
3806,Klay Thompson,Draymond Green,11.5,8.5,8.45,11.38,0.699,0.688,under,over,0,4.14,0.207,Med,Med
2725,Bilal Coulibaly,Deni Avdija,10.5,25.5,13.33,29.02,0.682,0.684,over,over,0,3.72,0.186,Med,High
3156,Josh Giddey,Aaron Gordon,20.5,17.5,23.22,19.75,0.660,0.666,over,over,0,2.93,0.146,High,Med
4794,Cameron Johnson,Shai Gilgeous-Alexander,11.5,30.5,9.26,28.10,0.660,0.658,under,under,0,2.77,0.139,Med,Med


### Prizepicks picks

In [7]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

prizepicksPairs = calculate2LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=15)


pairsPrizepicks = prizepicksPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'PROB 1', 'PROB 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
prizepicksPairs

Pre-computing predictions for 126 players...
Processing 115 players...
Generated 6230 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,PROB 1,PROB 2,PROB BOTH,EDGE 1,EDGE 2,COMBINED EDGE,EV$,KELLY FULL,RECOMMENDATION,SIGMA 1,SIGMA 2,SIGMA FLAG 1,SIGMA FLAG 2,CI 1,CI 2,CORRELATION,SAME_GAME,EXPECTED ROI
4633,Tre Jones,Dillon Brooks,8.5,16.5,14.89,22.80,over,over,0.864,0.830,0.7028,0.286,0.252,0.383,11.08,0.554,1,5.82,6.60,Med,High,"(3.5, 26.3)","(9.9, 35.7)",0.05,0,110.8
4282,Kevin Huerter,Alperen Sengun,12.5,22.5,16.73,28.14,over,over,0.752,0.807,0.5945,0.174,0.229,0.273,7.84,0.392,1,6.21,6.51,High,High,"(4.6, 28.9)","(15.4, 40.9)",0.05,0,78.4
3970,Coby White,Aaron Gordon,20.5,16.5,24.37,19.75,over,over,0.751,0.732,0.5390,0.173,0.154,0.216,6.17,0.309,0,5.71,5.24,Med,Med,"(13.2, 35.6)","(9.5, 30.0)",0.05,0,61.7
3268,Khris Middleton,Isaiah Collier,9.5,8.0,12.73,10.65,over,over,0.728,0.724,0.5170,0.150,0.146,0.193,5.51,0.275,0,5.31,4.45,Med,Low,"(2.3, 23.1)","(1.9, 19.4)",0.05,0,55.1
5489,Julius Randle,Lauri Markkanen,22.5,24.5,26.32,28.53,over,over,0.715,0.720,0.5041,0.137,0.142,0.180,5.12,0.256,0,6.74,6.92,High,High,"(13.1, 39.5)","(15.0, 42.1)",0.05,0,51.2
5249,Naji Marshall,Keyonte George,10.5,19.5,13.91,23.11,over,over,0.713,0.702,0.4904,0.135,0.124,0.166,4.71,0.236,0,6.06,6.82,High,High,"(2.0, 25.8)","(9.7, 36.5)",0.05,0,47.1
5105,Klay Thompson,Draymond Green,11.5,8.5,8.45,11.38,under,over,0.699,0.688,0.4714,0.121,0.110,0.147,4.14,0.207,0,5.86,5.86,Med,Med,"(0.0, 19.9)","(0.0, 22.9)",0.05,0,41.4
5051,D'Angelo Russell,Deni Avdija,12.5,25.5,15.66,29.02,over,over,0.684,0.684,0.4586,0.106,0.106,0.134,3.76,0.188,0,6.60,7.34,High,High,"(2.7, 28.6)","(14.6, 43.4)",0.05,0,37.6
3182,Bilal Coulibaly,Jerami Grant,10.5,19.5,13.33,17.10,over,under,0.682,0.675,0.4513,0.104,0.097,0.126,3.54,0.177,0,5.98,5.28,Med,Med,"(1.6, 25.0)","(6.8, 27.5)",0.05,0,35.4
591,Darius Garland,Brandon Williams,14.5,13.5,12.47,16.40,under,over,0.664,0.673,0.4379,0.086,0.095,0.113,3.14,0.157,0,4.78,6.47,Low,High,"(3.1, 21.8)","(3.7, 29.1)",0.05,0,31.4


## 3 leg parlay

### Underdog picks

In [8]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

underdogTrios = calculate3LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)

underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'PROB 1', 'PROB 2', 'PROB 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

Pre-computing predictions for 112 players...
Processing 103 players...
Generated 174972 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,PROB 1,PROB 2,PROB 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
138170,Tre Jones,Isaac Okoro,Dillon Brooks,8.5,6.5,17.5,14.89,11.22,22.80,0.864,0.785,0.789,over,over,over,1,18.91,0.378,Med,Med,High
36526,Bennedict Mathurin,Kevin Huerter,Alperen Sengun,21.5,12.5,23.5,26.11,16.73,28.14,0.761,0.752,0.762,over,over,over,1,13.54,0.271,High,High,High
2986,Darius Garland,Coby White,Lauri Markkanen,15.5,20.5,24.5,12.47,24.37,28.53,0.737,0.751,0.720,under,over,over,0,11.51,0.230,Low,Med,High
161583,Naji Marshall,Julius Randle,Keyonte George,10.5,22.5,19.5,13.91,26.32,23.11,0.713,0.715,0.702,over,over,over,0,9.31,0.186,High,High,High
120114,Bilal Coulibaly,Klay Thompson,Draymond Green,10.5,11.5,8.5,13.33,8.45,11.38,0.682,0.699,0.688,over,under,over,0,7.72,0.154,Med,Med,Med


### Prizepicks picks

In [9]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

triosPrizepicks = calculate3LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=15)


triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'PROB 1', 'PROB 2', 'PROB 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Pre-computing predictions for 126 players...
Processing 115 players...
Generated 244290 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,PROB 1,PROB 2,PROB 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
212117,Tre Jones,Dillon Brooks,Alperen Sengun,8.5,16.5,22.5,14.89,22.80,28.14,0.864,0.830,0.807,over,over,over,1,21.25,0.425,Med,High,High
188769,Coby White,Kevin Huerter,Aaron Gordon,20.5,12.5,16.5,24.37,16.73,19.75,0.751,0.752,0.732,over,over,over,0,12.33,0.247,Med,High,Med
163686,Khris Middleton,Isaiah Collier,Lauri Markkanen,9.5,8.0,24.5,12.73,10.65,28.53,0.728,0.724,0.720,over,over,over,0,10.50,0.210,Med,Low,High
228028,Naji Marshall,Julius Randle,Keyonte George,10.5,22.5,19.5,13.91,26.32,23.11,0.713,0.715,0.702,over,over,over,0,9.31,0.186,High,High,High
223037,D'Angelo Russell,Klay Thompson,Draymond Green,12.5,11.5,8.5,15.66,8.45,11.38,0.684,0.699,0.688,over,under,over,0,7.77,0.155,High,Med,Med


In [6]:
# playerScoring('Trey Murphy III', s26, current_date, teamStarPlayer, projectedStartingFive)
# playerContext('Trey Murphy III', s26, current_date, projectedStartingFive, mainStartingFive, teamStarPlayer)